In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

In [2]:
load_dotenv()

True

In [3]:
llm = init_chat_model(model="gemini-3.5-flash-lite", model_provider="google_genai")

In [4]:
class LLMstate(TypedDict):
    question : str
    answer : str

In [5]:
def essay_writer(state : LLMstate):
    question = state['question']
    state['answer'] = llm.invoke([HumanMessage(content=question)]).content
    return state

def essay_enhancer(state : LLMstate):
    question = f"Make this paragraph better {state['question']}"
    state['answer'] = llm.invoke([HumanMessage(content=question)]).content
    return state

In [6]:
graph = StateGraph(LLMstate)

graph.add_node('essay_writer', essay_writer)
graph.add_node('essay_enhancer', essay_enhancer)

graph.add_edge(START, 'essay_writer')
graph.add_edge('essay_writer', 'essay_enhancer')
graph.add_edge('essay_enhancer', END)

workflow = graph.compile()

In [7]:
initial_state = {'question': "Write an essay about 'The evolution of Computer'"}
final_state = workflow.invoke(initial_state)
print(final_state)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'question': "Write an essay about 'The evolution of Computer'", 'answer': [{'type': 'text', 'text': 'Here are a few ways to improve your prompt, depending on what you want to do next. \n\n### Option 1: If you want me to write the essay for you\n*Copy and paste one of these prompts:*\n* "Write a 500-word essay on the evolution of computers, starting from early mechanical calculators to modern quantum computing."\n* "Write an informative essay about the evolution of computers, focusing on how they went from room-sized machines to the smartphones we use today."\n\n---\n\n### Option 2: If you want to polish a paragraph *you* already wrote\n*Paste your paragraph below, and I will:*\n* Fix grammar and spelling.\n* Improve the flow and vocabulary.\n* Make the tone more academic or engaging.\n\n***\n\n**How would you like to proceed?**', 'extras': {'signature': 'El4KXAERTTIPdqZY2FIhQZXkLHVf+cB//6dG/vwgI0NStvhUoWDlIGQIdjCvbP4UsdGRePQQFUYix3A5PhqhuXj9zJAQMbBbBgnPQk4C3moyPHVLZGfSPR4u1N3mDbTm'}}]

In [8]:
print(final_state['answer'][0]['text'])

Here are a few ways to improve your prompt, depending on what you want to do next. 

### Option 1: If you want me to write the essay for you
*Copy and paste one of these prompts:*
* "Write a 500-word essay on the evolution of computers, starting from early mechanical calculators to modern quantum computing."
* "Write an informative essay about the evolution of computers, focusing on how they went from room-sized machines to the smartphones we use today."

---

### Option 2: If you want to polish a paragraph *you* already wrote
*Paste your paragraph below, and I will:*
* Fix grammar and spelling.
* Improve the flow and vocabulary.
* Make the tone more academic or engaging.

***

**How would you like to proceed?**
